# Loja do Seu Zé
Banco de dados PostgreSQL para gerenciar clientes, produtos, categorias, fornecedores, vendas, pagamentos e controle de estoque.

## Instalação e configuração do PostgreSQL no Colab

In [ ]:
# Instalação do PostgreSQL
!sudo apt update -qq
!sudo apt install -y -qq postgresql postgresql-contrib

In [ ]:
!pip install -q psycopg2-binary

In [ ]:
# Iniciar o servidor
!sudo service postgresql start

In [ ]:
!sudo service postgresql status

## Criação do banco de dados

In [ ]:
!sudo -u postgres createdb lojadoseuze

In [ ]:
# Atalho para rodar comandos SQL no banco lojadoseuze
!echo '#!/bin/bash' > /usr/local/bin/posql
!echo 'sudo -u postgres psql "$@" -d lojadoseuze & sudo -u postgres psql -c "\\q"' >> /usr/local/bin/posql
!chmod +x /usr/local/bin/posql

In [ ]:
!posql -c "select current_database();"

## Criando as tabelas

Ordem de criação respeita as dependências de chave estrangeira:

`categoria` → `cliente` → `fornecedor` → `produto` → `venda` → `item_venda` → `compra` → `pagamento_venda`

In [ ]:
%%bash
posql -c 'CREATE TABLE categoria(
    codigo_categoria INTEGER PRIMARY KEY,
    nome VARCHAR NOT NULL
)'

In [ ]:
%%bash
posql -c 'CREATE TABLE cliente(
    codigo_cliente INTEGER PRIMARY KEY,
    nome VARCHAR NOT NULL,
    cpf_cnpj VARCHAR UNIQUE NOT NULL,
    email VARCHAR UNIQUE NOT NULL
)'

In [ ]:
%%bash
posql -c 'CREATE TABLE fornecedor(
    codigo_fornecedor INTEGER PRIMARY KEY,
    razao_social VARCHAR NOT NULL,
    cnpj VARCHAR UNIQUE NOT NULL,
    telefone VARCHAR
)'

In [ ]:
%%bash
posql -c 'CREATE TABLE produto(
    codigo_produto INTEGER PRIMARY KEY,
    nome VARCHAR NOT NULL,
    preco_venda_atual DECIMAL NOT NULL,
    quantidade_estoque INTEGER NOT NULL,
    codigo_categoria INTEGER NOT NULL,
    CONSTRAINT fk_produto_categoria FOREIGN KEY (codigo_categoria)
        REFERENCES categoria(codigo_categoria)
)'

In [ ]:
%%bash
posql -c 'CREATE TABLE venda(
    codigo_venda INTEGER PRIMARY KEY,
    data_hora TIMESTAMP,
    valor_total DECIMAL,
    status_venda VARCHAR NOT NULL DEFAULT '\''PENDENTE'\'',
    codigo_cliente INTEGER,
    CONSTRAINT fk_venda_cliente FOREIGN KEY (codigo_cliente)
        REFERENCES cliente(codigo_cliente)
)'

In [ ]:
%%bash
posql -c 'CREATE TABLE item_venda(
    codigo_venda INTEGER NOT NULL,
    codigo_produto INTEGER NOT NULL,
    quantidade INTEGER NOT NULL,
    preco_unitario_gravado DECIMAL NOT NULL,
    PRIMARY KEY (codigo_venda, codigo_produto),
    CONSTRAINT fk_item_venda_venda FOREIGN KEY (codigo_venda)
        REFERENCES venda(codigo_venda),
    CONSTRAINT fk_item_venda_produto FOREIGN KEY (codigo_produto)
        REFERENCES produto(codigo_produto)
)'

In [ ]:
%%bash
posql -c 'CREATE TABLE compra(
    codigo_compra INTEGER PRIMARY KEY,
    codigo_fornecedor INTEGER NOT NULL,
    codigo_produto INTEGER NOT NULL,
    quantidade_comprada INTEGER NOT NULL,
    preco_custo_unitario DECIMAL NOT NULL,
    data_compra DATE NOT NULL,
    CONSTRAINT fk_compra_fornecedor FOREIGN KEY (codigo_fornecedor)
        REFERENCES fornecedor(codigo_fornecedor),
    CONSTRAINT fk_compra_produto FOREIGN KEY (codigo_produto)
        REFERENCES produto(codigo_produto)
)'

In [ ]:
%%bash
posql -c 'CREATE TABLE pagamento_venda(
    codigo_pagamento INTEGER PRIMARY KEY,
    codigo_venda INTEGER NOT NULL,
    forma_pagamento VARCHAR NOT NULL,
    valor_pago DECIMAL NOT NULL,
    CONSTRAINT fk_pagamento_venda FOREIGN KEY (codigo_venda)
        REFERENCES venda(codigo_venda)
)'

### Verificando as tabelas criadas

In [ ]:
!posql -c "\\d"

In [ ]:
!posql -c "\\d categoria"

In [ ]:
!posql -c "\\d cliente"

In [ ]:
!posql -c "\\d fornecedor"

In [ ]:
!posql -c "\\d produto"

In [ ]:
!posql -c "\\d venda"

In [ ]:
!posql -c "\\d item_venda"

In [ ]:
!posql -c "\\d compra"

In [ ]:
!posql -c "\\d pagamento_venda"

## Inserindo dados

A ordem de inserção segue a mesma lógica: tabelas referenciadas primeiro.

### Categorias

In [ ]:
%%bash
posql -c "INSERT INTO categoria VALUES (1, 'Bebida');"
posql -c "INSERT INTO categoria VALUES (2, 'Fruta');"
posql -c "INSERT INTO categoria VALUES (3, 'Cigarro');"
posql -c "INSERT INTO categoria VALUES (4, 'Salgado');"
posql -c "INSERT INTO categoria VALUES (5, 'Doce');"
posql -c "INSERT INTO categoria VALUES (6, 'Laticínio');"


In [ ]:
!posql -c "SELECT * FROM categoria;"

### Clientes

In [ ]:
%%bash
posql -c "INSERT INTO cliente VALUES (1, 'Cesar', '111.111.111-01', 'cesinha@gmail.com');"
posql -c "INSERT INTO cliente VALUES (2, 'Igor', '222.222.222-02', 'iguinho@gmail.com');"
posql -c "INSERT INTO cliente VALUES (3, 'Diogo', '333.333.333-03', 'dioguinho@gmail.com');"
posql -c "INSERT INTO cliente VALUES (4, 'Lazaro', '444.444.444-04', 'lazinho@gmail.com');"
posql -c "INSERT INTO cliente VALUES (5, 'Maria', '555.555.555-05', 'maria@gmail.com');"
posql -c "INSERT INTO cliente VALUES (6, 'João', '666.666.666-06', 'joao@gmail.com');"
posql -c "INSERT INTO cliente VALUES (7, 'Ana', '777.777.777-07', 'ana@gmail.com');"
posql -c "INSERT INTO cliente VALUES (8, 'Pedro', '888.888.888-08', 'pedro@gmail.com');"


In [ ]:
!posql -c "SELECT * FROM cliente;"

### Fornecedores

In [ ]:
%%bash
posql -c "INSERT INTO fornecedor VALUES (1, 'Distribuidora Norte', '11.111.111/0001-01', '84999990001');"
posql -c "INSERT INTO fornecedor VALUES (2, 'Tabacaria Central', '22.222.222/0001-02', '84999990002');"
posql -c "INSERT INTO fornecedor VALUES (3, 'Doces & Cia', '33.333.333/0001-03', '84999990003');"
posql -c "INSERT INTO fornecedor VALUES (4, 'Fazenda Boa Vista', '44.444.444/0001-04', '84999990004');"


In [ ]:
!posql -c "SELECT * FROM fornecedor;"

### Produtos

In [ ]:
%%bash
posql -c "INSERT INTO produto VALUES (1, 'Cafézinho', 2.00, 100, 1);"
posql -c "INSERT INTO produto VALUES (2, 'Maçã', 7.00, 10, 2);"
posql -c "INSERT INTO produto VALUES (3, 'Bad moro', 50.00, 100, 3);"
posql -c "INSERT INTO produto VALUES (4, 'Coxinha', 2.50, 5, 4);"
posql -c "INSERT INTO produto VALUES (5, 'Guaraná', 4.00, 50, 1);"
posql -c "INSERT INTO produto VALUES (6, 'Bolo de rolo', 5.00, 20, 5);"
posql -c "INSERT INTO produto VALUES (7, 'Queijo coalho', 3.50, 30, 6);"
posql -c "INSERT INTO produto VALUES (8, 'Pastel', 6.00, 15, 4);"
posql -c "INSERT INTO produto VALUES (9, 'Água', 2.00, 200, 1);"
posql -c "INSERT INTO produto VALUES (10, 'Paçoca', 1.50, 40, 5);"


In [ ]:
!posql -c "SELECT * FROM produto;"

### Vendas

In [ ]:
%%bash
posql -c "INSERT INTO venda VALUES (1, '2026-10-10 08:30:00', 6.50, 'PAGA', 1);"
posql -c "INSERT INTO venda VALUES (2, '2026-10-10 09:15:00', 52.00, 'PAGA', 3);"
posql -c "INSERT INTO venda VALUES (3, '2026-10-11 10:00:00', 25.00, 'PAGA', 2);"
posql -c "INSERT INTO venda VALUES (4, '2026-10-11 14:20:00', 7.00, 'PAGA', 4);"
posql -c "INSERT INTO venda VALUES (5, '2026-10-12 07:45:00', 9.00, 'PAGA', 5);"
posql -c "INSERT INTO venda VALUES (6, '2026-10-12 16:00:00', 14.00, 'PENDENTE', 1);"
posql -c "INSERT INTO venda VALUES (7, '2026-10-13 11:30:00', 9.50, 'PAGA', 6);"
posql -c "INSERT INTO venda VALUES (8, '2026-10-14 09:00:00', 6.50, 'CANCELADA', 7);"


In [ ]:
!posql -c "SELECT * FROM venda;"

### Itens das vendas

In [ ]:
%%bash
posql -c "INSERT INTO item_venda VALUES (1, 1, 2, 2.00);"   # 2 cafézinhos
posql -c "INSERT INTO item_venda VALUES (1, 4, 1, 2.50);"   # 1 coxinha
posql -c "INSERT INTO item_venda VALUES (2, 1, 1, 2.00);"   # 1 cafézinho
posql -c "INSERT INTO item_venda VALUES (2, 3, 1, 50.00);"  # 1 bad moro
posql -c "INSERT INTO item_venda VALUES (3, 2, 3, 7.00);"   # 3 maçãs
posql -c "INSERT INTO item_venda VALUES (3, 5, 1, 4.00);"   # 1 guaraná
posql -c "INSERT INTO item_venda VALUES (4, 4, 2, 2.50);"   # 2 coxinhas
posql -c "INSERT INTO item_venda VALUES (4, 9, 1, 2.00);"   # 1 água
posql -c "INSERT INTO item_venda VALUES (5, 7, 2, 3.50);"   # 2 queijo coalho
posql -c "INSERT INTO item_venda VALUES (5, 1, 1, 2.00);"   # 1 cafézinho
posql -c "INSERT INTO item_venda VALUES (6, 8, 1, 6.00);"   # 1 pastel
posql -c "INSERT INTO item_venda VALUES (6, 5, 2, 4.00);"   # 2 guaranás
posql -c "INSERT INTO item_venda VALUES (7, 6, 1, 5.00);"   # 1 bolo de rolo
posql -c "INSERT INTO item_venda VALUES (7, 10, 3, 1.50);"  # 3 paçocas
posql -c "INSERT INTO item_venda VALUES (8, 9, 2, 2.00);"   # 2 águas
posql -c "INSERT INTO item_venda VALUES (8, 4, 1, 2.50);"   # 1 coxinha


In [ ]:
!posql -c "SELECT * FROM item_venda;"

### Compras (abastecimento de estoque)

In [ ]:
%%bash
posql -c "INSERT INTO compra VALUES (1, 1, 1, 200, 0.80, '2026-10-01');"
posql -c "INSERT INTO compra VALUES (2, 1, 5, 100, 2.00, '2026-10-01');"
posql -c "INSERT INTO compra VALUES (3, 1, 9, 300, 0.50, '2026-10-01');"
posql -c "INSERT INTO compra VALUES (4, 2, 3, 150, 30.00, '2026-10-02');"
posql -c "INSERT INTO compra VALUES (5, 4, 2, 50, 3.00, '2026-10-03');"
posql -c "INSERT INTO compra VALUES (6, 4, 7, 60, 1.50, '2026-10-03');"
posql -c "INSERT INTO compra VALUES (7, 3, 6, 40, 2.50, '2026-10-04');"
posql -c "INSERT INTO compra VALUES (8, 3, 10, 80, 0.60, '2026-10-04');"
posql -c "INSERT INTO compra VALUES (9, 1, 4, 30, 1.00, '2026-10-05');"
posql -c "INSERT INTO compra VALUES (10, 1, 8, 40, 2.50, '2026-10-05');"


In [ ]:
!posql -c "SELECT * FROM compra;"

### Pagamentos

In [ ]:
%%bash
posql -c "INSERT INTO pagamento_venda VALUES (1, 1, 'PIX', 6.50);"
posql -c "INSERT INTO pagamento_venda VALUES (2, 2, 'DINHEIRO', 52.00);"
posql -c "INSERT INTO pagamento_venda VALUES (3, 3, 'PIX', 25.00);"
posql -c "INSERT INTO pagamento_venda VALUES (4, 4, 'DINHEIRO', 7.00);"
posql -c "INSERT INTO pagamento_venda VALUES (5, 5, 'PIX', 9.00);"
posql -c "INSERT INTO pagamento_venda VALUES (6, 7, 'DINHEIRO', 5.00);"
posql -c "INSERT INTO pagamento_venda VALUES (7, 7, 'PIX', 4.50);"


Vendas 6 (PENDENTE) e 8 (CANCELADA) não possuem pagamento. Venda 7 possui dois pagamentos (DINHEIRO + PIX) para demonstrar a relação 1:N.

In [ ]:
!posql -c "SELECT * FROM pagamento_venda;"

## Consultas de exemplo

### Listar todos os produtos com sua categoria

In [ ]:
%%bash
posql -c "SELECT p.nome AS produto, p.preco_venda_atual AS preco, c.nome AS categoria
FROM produto p
INNER JOIN categoria c ON p.codigo_categoria = c.codigo_categoria
ORDER BY c.nome, p.nome;"


### Detalhes de cada venda (cliente + itens)

In [ ]:
%%bash
posql -c "SELECT v.codigo_venda, cl.nome AS cliente, v.data_hora,
       p.nome AS produto, iv.quantidade, iv.preco_unitario_gravado,
       (iv.quantidade * iv.preco_unitario_gravado) AS subtotal
FROM venda v
INNER JOIN cliente cl ON v.codigo_cliente = cl.codigo_cliente
INNER JOIN item_venda iv ON v.codigo_venda = iv.codigo_venda
INNER JOIN produto p ON iv.codigo_produto = p.codigo_produto
ORDER BY v.codigo_venda;"


### Total de cada venda

In [ ]:
%%bash
posql -c "SELECT v.codigo_venda, cl.nome AS cliente,
       SUM(iv.quantidade * iv.preco_unitario_gravado) AS total
FROM venda v
INNER JOIN cliente cl ON v.codigo_cliente = cl.codigo_cliente
INNER JOIN item_venda iv ON v.codigo_venda = iv.codigo_venda
GROUP BY v.codigo_venda, cl.nome
ORDER BY v.codigo_venda;"


### Produto mais vendido (por quantidade)

In [ ]:
%%bash
posql -c "SELECT p.nome, SUM(iv.quantidade) AS total_vendido
FROM item_venda iv
INNER JOIN produto p ON iv.codigo_produto = p.codigo_produto
GROUP BY p.nome
ORDER BY total_vendido DESC;"


### Faturamento por categoria

In [ ]:
%%bash
posql -c "SELECT c.nome AS categoria,
       SUM(iv.quantidade * iv.preco_unitario_gravado) AS faturamento
FROM item_venda iv
INNER JOIN produto p ON iv.codigo_produto = p.codigo_produto
INNER JOIN categoria c ON p.codigo_categoria = c.codigo_categoria
GROUP BY c.nome
ORDER BY faturamento DESC;"


### Lucro por produto (venda vs custo)

In [ ]:
%%bash
posql -c "WITH custo_medio AS (
    SELECT codigo_produto,
           SUM(quantidade_comprada * preco_custo_unitario) / SUM(quantidade_comprada) AS custo_unitario_medio
    FROM compra
    GROUP BY codigo_produto
),
receita AS (
    SELECT iv.codigo_produto,
           SUM(iv.quantidade) AS qtd_vendida,
           SUM(iv.quantidade * iv.preco_unitario_gravado) AS receita_total
    FROM item_venda iv
    GROUP BY iv.codigo_produto
)
SELECT p.nome,
       r.qtd_vendida,
       ROUND(r.receita_total, 2) AS receita,
       ROUND(r.qtd_vendida * cm.custo_unitario_medio, 2) AS custo_total,
       ROUND(r.receita_total - (r.qtd_vendida * cm.custo_unitario_medio), 2) AS lucro
FROM receita r
INNER JOIN produto p ON r.codigo_produto = p.codigo_produto
INNER JOIN custo_medio cm ON r.codigo_produto = cm.codigo_produto
ORDER BY lucro DESC;"


### Pagamentos por forma de pagamento

In [ ]:
%%bash
posql -c "SELECT forma_pagamento,
       COUNT(*) AS qtd_pagamentos,
       SUM(valor_pago) AS total_recebido
FROM pagamento_venda
GROUP BY forma_pagamento
ORDER BY total_recebido DESC;"


### Fornecedores e seus produtos fornecidos

In [ ]:
%%bash
posql -c "SELECT f.razao_social AS fornecedor,
       p.nome AS produto,
       SUM(co.quantidade_comprada) AS total_fornecido,
       ROUND(AVG(co.preco_custo_unitario), 2) AS preco_custo_medio
FROM compra co
INNER JOIN fornecedor f ON co.codigo_fornecedor = f.codigo_fornecedor
INNER JOIN produto p ON co.codigo_produto = p.codigo_produto
GROUP BY f.razao_social, p.nome
ORDER BY f.razao_social, p.nome;"


## Aplicação — Simulação de uso real do BD

Usando `psycopg2` para conectar ao PostgreSQL e executar operações como uma aplicação faria.

In [ ]:
import psycopg2
from datetime import datetime

def conectar():
    return psycopg2.connect(
        dbname="lojadoseuze",
        user="postgres",
        host="/var/run/postgresql"
    )

conn = conectar()
conn.autocommit = False
print("Conectado ao banco lojadoseuze.")

### Funções da aplicação

Cada função simula uma operação real do sistema da loja.

In [ ]:
def consultar(sql, params=None):
    """Executa um SELECT e retorna os resultados."""
    with conn.cursor() as cur:
        cur.execute(sql, params)
        colunas = [desc[0] for desc in cur.description]
        linhas = cur.fetchall()
    return colunas, linhas


def mostrar(colunas, linhas):
    """Exibe resultados formatados."""
    larguras = [len(c) for c in colunas]
    for linha in linhas:
        for i, val in enumerate(linha):
            larguras[i] = max(larguras[i], len(str(val)))

    header = " | ".join(c.ljust(larguras[i]) for i, c in enumerate(colunas))
    separador = "-+-".join("-" * l for l in larguras)
    print(header)
    print(separador)
    for linha in linhas:
        print(" | ".join(str(v).ljust(larguras[i]) for i, v in enumerate(linha)))
    print(f"({len(linhas)} linhas)")

In [ ]:
def ver_estoque(codigo_produto=None):
    """Mostra o estoque atual. Se passar código, filtra um produto."""
    sql = """SELECT p.codigo_produto, p.nome, p.quantidade_estoque,
                    p.preco_venda_atual, c.nome AS categoria
             FROM produto p
             INNER JOIN categoria c ON p.codigo_categoria = c.codigo_categoria"""
    params = None
    if codigo_produto:
        sql += " WHERE p.codigo_produto = %s"
        params = (codigo_produto,)
    sql += " ORDER BY p.codigo_produto"
    mostrar(*consultar(sql, params))

In [ ]:
def registrar_compra(codigo_fornecedor, codigo_produto, quantidade, preco_custo):
    """Registra uma compra de fornecedor e atualiza o estoque."""
    try:
        with conn.cursor() as cur:
            # Gera próximo código
            cur.execute("SELECT COALESCE(MAX(codigo_compra), 0) + 1 FROM compra")
            codigo = cur.fetchone()[0]

            # Insere a compra
            cur.execute(
                """INSERT INTO compra
                   (codigo_compra, codigo_fornecedor, codigo_produto,
                    quantidade_comprada, preco_custo_unitario, data_compra)
                   VALUES (%s, %s, %s, %s, %s, %s)""",
                (codigo, codigo_fornecedor, codigo_produto,
                 quantidade, preco_custo, datetime.now().date())
            )

            # Atualiza estoque
            cur.execute(
                """UPDATE produto
                   SET quantidade_estoque = quantidade_estoque + %s
                   WHERE codigo_produto = %s""",
                (quantidade, codigo_produto)
            )

        conn.commit()
        print(f"Compra #{codigo} registrada: +{quantidade} unidades do produto {codigo_produto}")
        print(f"Custo unitário: R$ {preco_custo:.2f} | Total: R$ {quantidade * preco_custo:.2f}")
    except Exception as e:
        conn.rollback()
        print(f"ERRO na compra: {e}")

In [ ]:
def registrar_venda(codigo_cliente, itens, forma_pagamento):
    """
    Registra uma venda completa: venda + itens + pagamento + baixa no estoque.

    itens = lista de tuplas (codigo_produto, quantidade)
    forma_pagamento = 'PIX', 'DINHEIRO', etc.
    """
    try:
        with conn.cursor() as cur:
            # Verifica estoque antes de tudo
            for cod_prod, qtd in itens:
                cur.execute(
                    "SELECT nome, quantidade_estoque, preco_venda_atual FROM produto WHERE codigo_produto = %s",
                    (cod_prod,)
                )
                produto = cur.fetchone()
                if not produto:
                    raise ValueError(f"Produto {cod_prod} não encontrado.")
                nome, estoque, _ = produto
                if estoque < qtd:
                    raise ValueError(f"Estoque insuficiente para '{nome}': pedido {qtd}, disponível {estoque}.")

            # Gera código da venda
            cur.execute("SELECT COALESCE(MAX(codigo_venda), 0) + 1 FROM venda")
            cod_venda = cur.fetchone()[0]

            # Calcula valor total e insere itens
            valor_total = 0
            detalhes = []
            for cod_prod, qtd in itens:
                cur.execute("SELECT nome, preco_venda_atual FROM produto WHERE codigo_produto = %s", (cod_prod,))
                nome, preco = cur.fetchone()

                cur.execute(
                    "INSERT INTO item_venda VALUES (%s, %s, %s, %s)",
                    (cod_venda, cod_prod, qtd, preco)
                )

                subtotal = qtd * preco
                valor_total += subtotal
                detalhes.append(f"  {qtd}x {nome} @ R$ {preco:.2f} = R$ {subtotal:.2f}")

                # Baixa no estoque
                cur.execute(
                    "UPDATE produto SET quantidade_estoque = quantidade_estoque - %s WHERE codigo_produto = %s",
                    (qtd, cod_prod)
                )

            # Insere a venda
            cur.execute(
                "INSERT INTO venda VALUES (%s, %s, %s, %s, %s)",
                (cod_venda, datetime.now(), valor_total, 'PAGA', codigo_cliente)
            )

            # Insere o pagamento
            cur.execute("SELECT COALESCE(MAX(codigo_pagamento), 0) + 1 FROM pagamento_venda")
            cod_pag = cur.fetchone()[0]
            cur.execute(
                "INSERT INTO pagamento_venda VALUES (%s, %s, %s, %s)",
                (cod_pag, cod_venda, forma_pagamento, valor_total)
            )

        conn.commit()
        print(f"=== VENDA #{cod_venda} ===")
        print(f"Cliente: {codigo_cliente}")
        for d in detalhes:
            print(d)
        print(f"  TOTAL: R$ {valor_total:.2f}")
        print(f"  Pagamento: {forma_pagamento}")
        print("  Status: PAGA")
    except Exception as e:
        conn.rollback()
        print(f"ERRO na venda: {e}")

In [ ]:
def cancelar_venda(codigo_venda):
    """Cancela uma venda: muda status e devolve itens ao estoque."""
    try:
        with conn.cursor() as cur:
            cur.execute("SELECT status_venda FROM venda WHERE codigo_venda = %s", (codigo_venda,))
            resultado = cur.fetchone()
            if not resultado:
                raise ValueError(f"Venda {codigo_venda} não encontrada.")
            if resultado[0] == 'CANCELADA':
                raise ValueError(f"Venda {codigo_venda} já está cancelada.")

            # Devolve ao estoque
            cur.execute(
                "SELECT codigo_produto, quantidade FROM item_venda WHERE codigo_venda = %s",
                (codigo_venda,)
            )
            for cod_prod, qtd in cur.fetchall():
                cur.execute(
                    "UPDATE produto SET quantidade_estoque = quantidade_estoque + %s WHERE codigo_produto = %s",
                    (qtd, cod_prod)
                )

            # Remove pagamentos e muda status
            cur.execute("DELETE FROM pagamento_venda WHERE codigo_venda = %s", (codigo_venda,))
            cur.execute("UPDATE venda SET status_venda = 'CANCELADA' WHERE codigo_venda = %s", (codigo_venda,))

        conn.commit()
        print(f"Venda #{codigo_venda} cancelada. Estoque devolvido.")
    except Exception as e:
        conn.rollback()
        print(f"ERRO ao cancelar: {e}")

In [ ]:
def relatorio_lucro():
    """Relatório de lucro por produto (receita - custo)."""
    sql = """
    WITH custo_medio AS (
        SELECT codigo_produto,
               SUM(quantidade_comprada * preco_custo_unitario) / SUM(quantidade_comprada) AS custo_unit
        FROM compra
        GROUP BY codigo_produto
    ),
    receita AS (
        SELECT iv.codigo_produto,
               SUM(iv.quantidade) AS qtd_vendida,
               SUM(iv.quantidade * iv.preco_unitario_gravado) AS receita_total
        FROM item_venda iv
        INNER JOIN venda v ON iv.codigo_venda = v.codigo_venda
        WHERE v.status_venda != 'CANCELADA'
        GROUP BY iv.codigo_produto
    )
    SELECT p.nome,
           r.qtd_vendida,
           ROUND(r.receita_total, 2) AS receita,
           ROUND(r.qtd_vendida * cm.custo_unit, 2) AS custo,
           ROUND(r.receita_total - (r.qtd_vendida * cm.custo_unit), 2) AS lucro
    FROM receita r
    INNER JOIN produto p ON r.codigo_produto = p.codigo_produto
    INNER JOIN custo_medio cm ON r.codigo_produto = cm.codigo_produto
    ORDER BY lucro DESC
    """
    mostrar(*consultar(sql))

### Demonstração

#### 1. Ver estoque atual

In [ ]:
ver_estoque()

#### 2. Registrar uma compra de fornecedor (abastecer estoque)

In [ ]:
# Comprando 50 coxinhas da Distribuidora Norte a R$ 1.00 cada
registrar_compra(
    codigo_fornecedor=1,
    codigo_produto=4,
    quantidade=50,
    preco_custo=1.00
)

In [ ]:
# Conferir estoque do produto 4 (Coxinha)
ver_estoque(4)

#### 3. Registrar uma venda

In [ ]:
# Cliente 3 (Diogo) compra 2 cafézinhos + 3 coxinhas, paga no PIX
registrar_venda(
    codigo_cliente=3,
    itens=[(1, 2), (4, 3)],
    forma_pagamento='PIX'
)

In [ ]:
# Estoque atualizado
ver_estoque()

#### 4. Tentar vender mais do que tem no estoque

In [ ]:
# Tentar vender 999 maçãs — deve dar erro
registrar_venda(
    codigo_cliente=1,
    itens=[(2, 999)],
    forma_pagamento='DINHEIRO'
)

#### 5. Cancelar uma venda

In [ ]:
# Cancelar a venda 6 (que estava PENDENTE)
cancelar_venda(6)

In [ ]:
# Conferir que o estoque voltou
ver_estoque()

#### 6. Relatório de lucro

In [ ]:
relatorio_lucro()

#### 7. Encerrar conexão

In [ ]:
conn.close()
print('Conexão encerrada.')